# Análisis de resultados — por qué el modelo elegido es el que es

Notebook de consulta: **lee la tabla destino y no escribe nada**. No recalcula forecasts; toma lo
que el pipeline ya dejó guardado y reconstruye la decisión.

La clave: el score se recalcula con `evaluate_models()`, **la misma función que usa el motor para
elegir**. No es una reimplementación parecida, es el mismo código — así los números que ves acá
son los que decidieron la elección, no una aproximación.

Recorrido:

1. Panorama de la última corrida
2. Qué series conviene mirar (peor error, elección ajustada, sesgo alto)
3. Una serie en detalle, tal cual está en Oracle
4. **Por qué ganó ese modelo**: ranking completo con su error
5. Mes a mes: dónde perdió cada modelo
6. El gráfico
7. Historial: cómo fue cambiando el modelo elegido

In [ ]:
# ── Qué querés mirar ─────────────────────────────────────────────────────────
SERIE = None          # None = la peor por error. O fijala: {"SK_CLIENTE": 900001}
MESES = None          # None = toda la tabla. O un número de meses hacia atrás.
FILTRO = None         # predicado SQL extra, ej. "SK_CLIENTE IN (900001, 900002)"
TOP = 15              # cuántas filas mostrar en los rankings

## Setup

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import fc_oracle as io
from forecast_engine import evaluate_models, detect_model_cols

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

cfg = io.build_config()
CATS = [c.upper() for c in cfg.category_cols]
FECHA = io.FECHA_DB
COL = io.COLUMNAS_FIJAS

with io.conexion_destino() as conn:
    tabla = io.leer_tabla_destino(conn, cfg, meses=MESES, filtro=FILTRO)

if tabla.empty:
    raise RuntimeError(f"{io.TABLA_SALIDA} no devolvió filas con ese filtro")

motor = io.a_nombres_motor(tabla, cfg)              # mismos datos, nombres del motor
modelos = detect_model_cols(motor, cfg)             # {columna: modelo}
corte = tabla[COL["corte"]].max()                   # corte de la última corrida
print(f"{len(tabla):,} filas | {tabla.groupby(CATS).ngroups:,} series | "
      f"{tabla[FECHA].min():%Y-%m} a {tabla[FECHA].max():%Y-%m}")
print(f"última corrida: corte {corte:%Y-%m-%d} | ventana de selección: "
      f"{cfg.resolved_score_window()} meses | métrica: {cfg.metric} "
      f"+ {cfg.bias_weight} × sesgo")

## 1. Panorama

`IS_FUTURE = 0` son los meses ya cerrados (tienen el real al lado) y `1` la proyección.
El wMAPE de cada mes es el error del **modelo elegido** contra el real de ese mes.

In [ ]:
hist = tabla[tabla[COL["futuro"]] == 0].copy()
hist["err_abs"] = (hist[COL["ganador"]] - hist[COL["real"]]).abs()

por_mes = hist.groupby(FECHA).agg(series=(CATS[0], "nunique"),
                                  real=(COL["real"], "sum"),
                                  error=("err_abs", "sum"))
por_mes["wmape_%"] = 100 * por_mes.error / por_mes.real.replace(0, np.nan)
por_mes["en_ventana"] = por_mes.index > (corte - pd.DateOffset(months=cfg.resolved_score_window()))
print(f"wMAPE global (todo el histórico guardado): "
      f"{100 * hist.err_abs.sum() / hist[COL['real']].sum():.2f}%")
en_v = hist[hist[FECHA] > corte - pd.DateOffset(months=cfg.resolved_score_window())]
print(f"wMAPE en la ventana de selección: "
      f"{100 * en_v.err_abs.sum() / en_v[COL['real']].sum():.2f}%")
por_mes

In [ ]:
# Modelos elegidos en la última corrida (mirando las filas futuras, que son las que valen hoy)
futuro = tabla[tabla[COL["futuro"]] == 1]
(futuro.groupby(COL["modelo_ganador"])[CATS[0]].nunique()
       .sort_values(ascending=False)
       .rename("series").to_frame()
       .assign(**{"%": lambda d: 100 * d.series / d.series.sum()}))

## 2. Qué series mirar

Tres listas distintas, porque "malo" no es una sola cosa:

- **peor error**: el modelo elegido igual le erra mucho — puede que ningún modelo sirva para esa serie.
- **elección ajustada**: el ganador le sacó muy poco al segundo, así que la elección es frágil y
  puede darse vuelta el mes que viene. No es un problema en sí, pero explica los cambios de modelo.
- **sesgo alto**: el ganador acierta la magnitud pero se queda sistemáticamente corto o largo.

In [ ]:
scores = evaluate_models(motor, cfg, modelos, cutoff=corte)
if scores.empty:
    raise RuntimeError("No hay puntos evaluables en la ventana: revisá el rango leído (MESES)")

scores = scores.sort_values(CATS + ["score"])
rank = scores.groupby(CATS).cumcount() + 1
scores["puesto"] = rank
ganadores = scores[scores.puesto == 1].set_index(CATS)
segundos = scores[scores.puesto == 2].set_index(CATS)

comparativa = pd.DataFrame({
    "modelo": ganadores.model,
    "wmape_%": 100 * ganadores.wmape,
    "sesgo_%": 100 * ganadores.bias / ganadores.mae.replace(0, np.nan),
    "score": ganadores.score,
    "segundo": segundos.model,
    "score_2do": segundos.score,
    "puntos": ganadores.n_points,
})
comparativa["ventaja_%"] = 100 * (comparativa.score_2do - comparativa.score) / comparativa.score_2do

print("── peor error ──")
display(comparativa.sort_values("wmape_%", ascending=False).head(TOP))
print("── elección más ajustada (el 2º casi gana) ──")
display(comparativa.sort_values("ventaja_%").head(TOP))
print("── más sesgo (+ se pasa, − se queda corto) ──")
display(comparativa.reindex(comparativa["sesgo_%"].abs().sort_values(ascending=False).index).head(TOP))

## 3. La serie en detalle

Si dejaste `SERIE = None` toma la de peor error. Esto es la tabla de Oracle tal cual, sin
transformar: una fila por mes y una columna por modelo.

In [ ]:
if SERIE:
    clave = tuple(SERIE[c] for c in CATS)
else:
    clave = comparativa.sort_values("wmape_%", ascending=False).index[0]
    clave = clave if isinstance(clave, tuple) else (clave,)

mask = np.logical_and.reduce([tabla[c] == v for c, v in zip(CATS, clave)])
serie = tabla[mask].sort_values(FECHA)
print("serie:", dict(zip(CATS, clave)), "|", len(serie), "meses guardados")
serie

## 4. Por qué ganó ese modelo

Cada modelo se puntúa sobre la **misma ventana móvil** (los últimos `score_window` meses cerrados):

```
score = wMAPE ponderado por recencia  +  bias_weight × |sesgo| / nivel
```

- **wmape_%**: error absoluto medio como % del nivel de la serie. Es el grueso del score.
- **sesgo_%**: error medio con signo, como % del error absoluto medio. Cerca de ±100% significa
  que el modelo *siempre* se equivoca para el mismo lado; cerca de 0 que se compensa.
- **score**: lo que decide. Gana el más bajo.
- **puntos**: cuántos meses de la ventana pudo evaluar ese modelo (si son menos que el resto, el
  modelo falló en algunos meses).

Abajo se compara el ganador recalculado contra el `BEST_MODEL`/`BEST_SCORE` guardados. En una
corrida mensual dan idénticos, porque el motor puntúa leyendo de esta misma tabla. Puede haber una
diferencia en el último decimal si la fila viene de la corrida inicial, donde el motor puntuó con
precisión completa y recién después redondeó a `round_to` para escribir.

In [ ]:
det = scores.loc[np.logical_and.reduce([scores[c] == v for c, v in zip(CATS, clave)])].copy()
det = det.sort_values("score")
det["sesgo_%"] = 100 * det.bias / det.mae.replace(0, np.nan)
det["wmape_%"] = 100 * det.wmape
guardado = serie[serie[COL["futuro"]] == 1]
gan_guardado = guardado[COL["modelo_ganador"]].iloc[0] if len(guardado) else None
score_guardado = guardado[COL["score"]].iloc[0] if len(guardado) else np.nan

vista = det[["puesto", "model", "wmape_%", "sesgo_%", "score", "n_points", "coverage"]].copy()
vista.insert(0, "elegido", np.where(det.model == gan_guardado, "<<<", ""))
print(f"guardado en la tabla: BEST_MODEL = {gan_guardado} | BEST_SCORE = {score_guardado:.6f}")
print(f"recalculado ahora   : {det.model.iloc[0]} | score = {det.score.iloc[0]:.6f} "
      f"| dif = {abs(det.score.iloc[0] - score_guardado):.6f}")
if det.model.iloc[0] != gan_guardado:
    print("\n[!] el ganador recalculado NO es el guardado. Suele ser porque la tabla se leyó con "
          "menos meses de los que usa la ventana (subí MESES) o porque la fila viene de una "
          "corrida con otra configuración de modelos.")
vista.reset_index(drop=True)

## 5. Mes a mes: dónde perdió cada modelo

Error con signo (`forecast − real`) de cada modelo en cada mes de la ventana. Sirve para ver si un
modelo perdió por un mes puntual —una promo, un mes raro— o porque viene errando siempre.
La fila `real` está para dimensionar.

In [ ]:
ventana_ini = corte - pd.DateOffset(months=cfg.resolved_score_window() - 1)
w = serie[(serie[FECHA] >= ventana_ini) & (serie[FECHA] <= corte)].set_index(FECHA)

errores = pd.DataFrame({m: w[io.COLUMNAS_MODELO[m]] - w[COL["real"]]
                        for m in det.model if m in io.COLUMNAS_MODELO})
errores.insert(0, "real", w[COL["real"]])
errores.index = errores.index.strftime("%Y-%m")
print("error con signo por mes (filas = modelos, en orden del ranking)")
errores.T

## 6. El gráfico

In [ ]:
top = list(det.model.head(4))
fig, ax = plt.subplots(figsize=(12, 5))
s = serie.set_index(FECHA)
ax.plot(s.index, s[COL["real"]], "o-", color="#111", lw=2.5, label="real", zorder=5)
for m, color in zip(top, ["#c2410c", "#0369a1", "#15803d", "#7c3aed"]):
    col = io.COLUMNAS_MODELO.get(m)
    if col in s.columns:
        estilo = "-" if m == gan_guardado else "--"
        ax.plot(s.index, s[col], estilo, color=color, alpha=.9,
                label=f"{m}{'  ← elegido' if m == gan_guardado else ''}")
ax.axvline(corte, color="#64748b", ls=":", lw=1.5)
ax.axvspan(ventana_ini, corte, color="#94a3b8", alpha=.12)
ax.annotate("ventana de selección", (ventana_ini, ax.get_ylim()[1]), color="#475569",
            va="bottom", fontsize=9)
ax.annotate("corte", (corte, ax.get_ylim()[0]), color="#475569", fontsize=9)
ax.set_title(f"{dict(zip(CATS, clave))} — real vs. los 4 mejores modelos")
ax.legend(loc="upper left", fontsize=9); ax.grid(alpha=.25)
fig.tight_layout(); plt.show()

## 7. Historial: cómo fue cambiando el modelo elegido

Cada fila de la tabla guarda el `BEST_MODEL` de la corrida que la escribió, y esas filas no se
reescriben. Así que la columna es el registro de qué se decidió en cada momento.

Cambiar de modelo no es malo en sí: significa que entró un mes nuevo a la ventana y movió el
ranking. Vale la pena mirarlo cuando una serie cambia todos los meses — ahí la elección está
empatada y conviene fijar el modelo a mano o usar `combo_median`.

In [ ]:
hist_serie = (serie[[FECHA, COL["modelo_ganador"], COL["score"], COL["corte"], COL["run"]]]
              .drop_duplicates(subset=[COL["corte"]]).sort_values(FECHA))
print("elección por corrida, para esta serie:")
display(hist_serie)

cambios = (tabla[tabla[COL["futuro"]] == 0]
           .sort_values(CATS + [FECHA])
           .assign(anterior=lambda d: d.groupby(CATS)[COL["modelo_ganador"]].shift()))
cambios = cambios[cambios.anterior.notna() &
                  (cambios.anterior != cambios[COL["modelo_ganador"]])]
print(f"\nseries que cambiaron de modelo alguna vez: "
      f"{cambios.groupby(CATS).ngroups} de {tabla.groupby(CATS).ngroups}")
(cambios.groupby(CATS).size().sort_values(ascending=False).head(TOP)
        .rename("veces que cambió de modelo").to_frame())

## Cómo leer todo esto

**El ganador tiene mal wMAPE.** Ningún modelo le pega a esa serie. Mirá el gráfico: si el real
salta sin patrón, es ruido y no hay modelo que lo arregle — el valor de la elección es que al menos
no exagera. Si en cambio ves un quiebre de nivel, `prophet` debería estar arriba en el ranking; si
no lo está, revisá que tenga suficientes meses.

**Ganó por poquito.** Elección frágil, va a rotar. Si te molesta la inestabilidad, `combo_median`
suele quedar segundo o tercero siempre y casi no rota.

**Sesgo cerca de ±100%.** El modelo va todos los meses para el mismo lado. Si el ganador está así,
mirá si algún competidor con wMAPE parecido tiene menos sesgo: subir `bias_weight` en
`build_config()` lo favorecería.

**Pocos `puntos` en un modelo.** Falló en algunos meses de la ventana (series cortas, muchos ceros,
o no convergió). Con menos de `min_coverage` de cobertura queda directamente fuera del ranking.